# ICE Outcomes: Data Cleaning, Feature Engineering, and EDA

In [179]:
# imports
import pandas as pd
import numpy as np
import re

### Data Loading
---
Initial setup - load dataset, inspect column names, data types, duplicates, and blank columns.

In [180]:
# Load the CSV from your local path
df = pd.read_csv("../data/ICE_Master_Cleaned.csv")

# Print all column names
print("Column Names:")
print(df.columns.tolist())

# Total column count
print(f"\nTotal columns: {len(df.columns)}")

# Column names with data types
print("\nColumn names and data types:")
print(df.dtypes)

# Check for duplicate column names
dupes = df.columns[df.columns.duplicated()].tolist()
print(f"\nDuplicate columns: {dupes if dupes else 'None'}")

# Check for unnamed/blank columns
unnamed = [col for col in df.columns if 'Unnamed' in str(col) or str(col).strip() == '']
print(f"Unnamed/blank columns: {unnamed if unnamed else 'None'}")

/var/folders/_s/x14mjmjn7qj5qh8bhxln436r0000gn/T/ipykernel_34384/547015630.py:2: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,16,17,18,19,20,21,23,24,26,27,28,29,30,31,33,34,35,36,37,38,39,40,41,42,43,44,47,48,49,50,51,52,54,55,56,57) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/ICE_Master_Cleaned.csv")


Column Names:
['detentions_Case Category', 'detentions_Bond Posted Date', 'detentions_Detention Book Out Date', 'removals_Entry Status', 'detentions_Final Order Yes No', 'detentions_Ethnicity', 'removals_Departure Country', 'removals_Alien File Number', 'detentions_Charge Code', 'removals_Departure Date', 'removals_MSC Conviction Date', 'detentions_Stay Book In Date', 'removals_Anonymized Identifer', 'detentions_Anonymized Identifier', 'detentions_Birth Date', 'Anonymized Identifier', 'removals_Case Status', 'removals_Citizenship Country', 'detentions_Detention Facility Code', 'arrests_Apprehension Method', 'detentions_Religion', 'detentions_Detention Facility', 'detentions_Bond Posted Amount', 'detentions_Detention Release Reason', 'detentions_Final Order Date', 'removals_Birth Year', 'removals_Gender', 'removals_Entry Date', 'arrests_Apprehension Date', 'detentions_Departure Country', 'detentions_Stay Release Reason', 'removals_Birth Date', 'removals_Case Threat Level', 'arrests_Anon

In [181]:
df.head()

,detentions_Case Category,detentions_Bond Posted Date,detentions_Detention Book Out Date,removals_Entry Status,detentions_Final Order Yes No,detentions_Ethnicity,removals_Departure Country,removals_Alien File Number,detentions_Charge Code,removals_Departure Date,...,detentions_Stay Book Out Date,removals_Port of Departure,arrests_Case ID,removals_Birth Country,detentions_Birth Year,detentions_Departed Date,detentions_Gender,detentions_Case Status,removals_Case Category,year
0,[5F] Unable to Obtain Travel Document,1992-07-14,2012-09-28 00:00:00,NaN,True,Not of Hispanic Origin,NaN,NaN,NaN,NaN,...,2012-09-28 00:00:00,NaN,(b)(6)(b)(7)(c),NaN,1969.0,NaN,Male,ACTIVE,NaN,2012
1,[8B] Excludable / Inadmissible - Under Adjudic...,2012-06-20,2012-06-19 00:00:00,NaN,False,Hispanic Origin,NaN,NaN,NaN,NaN,...,2012-06-20 00:00:00,NaN,(b)(6)(b)(7)(c),NaN,1979.0,NaN,Male,ACTIVE,NaN,2012
2,[8C] Excludable / Inadmissible - Administrativ...,2013-02-11,2013-02-11 00:00:00,NaN,True,Hispanic Origin,NaN,NaN,NaN,NaN,...,2013-02-11 00:00:00,NaN,NaN,NaN,1974.0,NaN,Male,B-Relief Granted,NaN,2012
3,[5B] Removable - ICE Fugitive,2012-08-07,2012-07-05 00:00:00,NaN,True,NaN,NaN,NaN,NaN,NaN,...,2012-08-07 00:00:00,NaN,NaN,NaN,1974.0,NaN,Male,ACTIVE,NaN,2012
4,[8E] Inadmissible - ICE Fugitive,NaN,2012-11-19 00:00:00,NaN,True,Hispanic Origin,NaN,NaN,NaN,NaN,...,2012-11-19 00:00:00,NaN,NaN,NaN,1973.0,NaN,Male,ACTIVE,NaN,2012


In [182]:
print(df.columns.tolist())

['detentions_Case Category', 'detentions_Bond Posted Date', 'detentions_Detention Book Out Date', 'removals_Entry Status', 'detentions_Final Order Yes No', 'detentions_Ethnicity', 'removals_Departure Country', 'removals_Alien File Number', 'detentions_Charge Code', 'removals_Departure Date', 'removals_MSC Conviction Date', 'detentions_Stay Book In Date', 'removals_Anonymized Identifer', 'detentions_Anonymized Identifier', 'detentions_Birth Date', 'Anonymized Identifier', 'removals_Case Status', 'removals_Citizenship Country', 'detentions_Detention Facility Code', 'arrests_Apprehension Method', 'detentions_Religion', 'detentions_Detention Facility', 'detentions_Bond Posted Amount', 'detentions_Detention Release Reason', 'detentions_Final Order Date', 'removals_Birth Year', 'removals_Gender', 'removals_Entry Date', 'arrests_Apprehension Date', 'detentions_Departure Country', 'detentions_Stay Release Reason', 'removals_Birth Date', 'removals_Case Threat Level', 'arrests_Anonymized Identif

In [183]:
null_count = df['detentions_Final Order Yes No'].isnull().sum()
print(f"Number of nulls in detentions_Final Order Yes No: {null_count}")

Number of nulls in detentions_Final Order Yes No: 315511


In [184]:
print(f"Unique values in detentions_Final Order Yes No: {df['detentions_Final Order Yes No'].unique()}")

Unique values in detentions_Final Order Yes No: [True False nan]


### Target Variable Cleaning
---
In our case, we want to predict whether someone received a final deportation order. The column detentions_Final Order Yes No is our target feature. 

In [185]:
# dropping rows where detentions_Final Order is null since we can't determine the outcome for those cases
df = df.dropna(subset=['detentions_Final Order Yes No'])

In [186]:
null_count = df['detentions_Final Order Yes No'].isnull().sum()
print(f"Number of nulls in detentions_Final Order Yes No: {null_count}")

Number of nulls in detentions_Final Order Yes No: 0


In [187]:
df['detentions_Final_Order_numeric'] = df['detentions_Final Order Yes No'].map({
    True: 1,
    False: 0
})

### Handle FOIA Redactions
---
This dataset was obtained through a Freedom of Information Act (FOIA) request, which means the government released it to the public but was legally required to black out certain personal identifying information. Instead of actually blacking out the text, the dataset uses the code (b)(6)(b)(7)(c) wherever information was withheld. This shows up all over columns like names, case IDs, and alien file numbers. These values look like real data but are actually just a placeholder meaning "redacted." If we left them in, the model might treat them as a meaningful category, which would be misleading. So we replace every instance of this redaction code with a true missing value (NaN) so Python knows the data isn't there.
- Replace all FOIA redacted values (b)(6)(b)(7)(c) with NaN for accurate modeling.


In [188]:
# HANDLE FOIA REDACTIONS
foia_pattern = r'\(b\)\(\d+\)'

# Only look at object columns, but explicitly cast to string first
# because some "object" columns contain mixed types (numbers + text)
# which causes .str to fail
obj_cols = df.select_dtypes(include='object').columns

for col in obj_cols:
    # Convert to string temporarily just for the check
    # na=False means NaN values are ignored and treated as non-matches
    col_as_str = df[col].astype(str)
    mask = col_as_str.str.contains(foia_pattern, regex=True, na=False)
    if mask.sum() > 0:
        df.loc[mask, col] = np.nan
        print(f"  Replaced {mask.sum()} redacted values in: {col}")

print(f"\nDataset shape after FOIA cleaning: {df.shape}")


  Replaced 1118967 redacted values in: removals_Alien File Number
  Replaced 2194212 redacted values in: detentions_Birth Date
  Replaced 1118962 redacted values in: removals_Birth Date
  Replaced 1034698 redacted values in: arrests_Alien File Number
  Replaced 1034698 redacted values in: arrests_Subject ID
  Replaced 1028072 redacted values in: arrests_Case ID

Dataset shape after FOIA cleaning: (2194212, 60)


In [189]:
print(df.columns.tolist())

['detentions_Case Category', 'detentions_Bond Posted Date', 'detentions_Detention Book Out Date', 'removals_Entry Status', 'detentions_Final Order Yes No', 'detentions_Ethnicity', 'removals_Departure Country', 'removals_Alien File Number', 'detentions_Charge Code', 'removals_Departure Date', 'removals_MSC Conviction Date', 'detentions_Stay Book In Date', 'removals_Anonymized Identifer', 'detentions_Anonymized Identifier', 'detentions_Birth Date', 'Anonymized Identifier', 'removals_Case Status', 'removals_Citizenship Country', 'detentions_Detention Facility Code', 'arrests_Apprehension Method', 'detentions_Religion', 'detentions_Detention Facility', 'detentions_Bond Posted Amount', 'detentions_Detention Release Reason', 'detentions_Final Order Date', 'removals_Birth Year', 'removals_Gender', 'removals_Entry Date', 'arrests_Apprehension Date', 'detentions_Departure Country', 'detentions_Stay Release Reason', 'removals_Birth Date', 'removals_Case Threat Level', 'arrests_Anonymized Identif

### Drop ID / Unnecessary Columns / Redacted Columns
---

The following columns were removed from the dataset because they fall into one of these categories:
 
1. **Columns with high missing values** – columns where a large portion of the data is missing, making them unreliable for modeling.  
2. **Administrative / metadata columns** – including names, officer IDs, submission dates, and other non-informative fields unlikely to contribute to analysis or model performance.

These steps help simplify the dataset and focus on features that are relevant and usable for feature engineering and predictive modeling.

In [190]:
df.isna().sum().sort_values(ascending=False)

arrests_Subject ID                     2194212
removals_Alien File Number             2194212
removals_Birth Date                    2194212
arrests_Case ID                        2194212
detentions_Birth Date                  2194212
arrests_Alien File Number              2194212
detentions_Religion                    2110916
detentions_Bond Posted Date            1846747
detentions_Bond Posted Amount          1846742
detentions_Initial Bond Set Amount     1808799
removals_Case Threat Level             1537447
removals_MSC Charge Date               1517952
removals_Entry Date                    1507428
removals_MSC Conviction Date           1497134
removals_MSC Charge                    1497134
removals_MSC Charge Code               1497134
detentions_Ethnicity                   1178947
arrests_Apprehension Date              1159514
arrests_Apprehension Method            1159514
arrests_Anonymized Identifier          1159514
removals_Apprehension Date             1142950
removals_Fina

In [191]:
# COLUMNS TO DROP 
high_missing_cols = [
    'arrests_Subject ID',
    'removals_Alien File Number',
    'removals_Birth Date',
    'arrests_Case ID',
    'detentions_Birth Date',
    'arrests_Alien File Number',
    'detentions_Religion',
    'detentions_Bond Posted Date',
    'detentions_Bond Posted Amount',
    'detentions_Initial Bond Set Amount',
    'removals_Case Threat Level',
    'removals_MSC Charge Date',
    'removals_Entry Date',
    'removals_MSC Conviction Date',
    'removals_MSC Charge',
    'removals_MSC Charge Code',
    'arrests_Anonymized Identifier',
    'removals_Final Order Date',
    'removals_Entry Status',
    'removals_Departure Country',
    'removals_Port of Departure',
    'removals_Departure Date',
    'removals_Final Order Yes No',
    'removals_Case Status',
    'removals_Case Category',
    'removals_Processing Disposition',
    'removals_Birth Year',
    'removals_Gender',
    'removals_Citizenship Country',
    'removals_Birth Country',
    'removals_Anonymized Identifer',
    'detentions_Charge Code',
    'detentions_Charge',
    'detentions_Final Order Date',
    'arrests_Apprehension Date',
    'removals_Apprehension Date'
]

df_cleaned = df.drop(columns=[col for col in high_missing_cols if col in df.columns])

In [192]:
df_cleaned.shape

(2194212, 24)

In [193]:
df_cleaned.isna().sum().sort_values(ascending=False)

detentions_Ethnicity                   1178947
arrests_Apprehension Method            1159514
detentions_Case Threat Level           1026160
detentions_Departure Country            641313
detentions_Departed Date                639668
detentions_Entry Status                  82518
detentions_Marital                       59333
detentions_Stay Release Reason            2895
detentions_Stay Book Out Date             2895
detentions_Detention Book Out Date         632
detentions_Detention Release Reason        632
detentions_Birth Year                        3
detentions_Case Category                     2
detentions_Detention Facility                0
detentions_Detention Facility Code           0
Anonymized Identifier                        0
detentions_Detention Book In Date            0
detentions_Anonymized Identifier             0
detentions_Stay Book In Date                 0
detentions_Final Order Yes No                0
detentions_Gender                            0
detentions_Ca

In [194]:
print(df_cleaned.columns.tolist())

['detentions_Case Category', 'detentions_Detention Book Out Date', 'detentions_Final Order Yes No', 'detentions_Ethnicity', 'detentions_Stay Book In Date', 'detentions_Anonymized Identifier', 'Anonymized Identifier', 'detentions_Detention Facility Code', 'arrests_Apprehension Method', 'detentions_Detention Facility', 'detentions_Detention Release Reason', 'detentions_Departure Country', 'detentions_Stay Release Reason', 'detentions_Marital', 'detentions_Entry Status', 'detentions_Case Threat Level', 'detentions_Detention Book In Date', 'detentions_Stay Book Out Date', 'detentions_Birth Year', 'detentions_Departed Date', 'detentions_Gender', 'detentions_Case Status', 'year', 'detentions_Final_Order_numeric']


**Handle Missing Values**

In [195]:
# Categorical columns
categorical_cols = [
    'detentions_Ethnicity', 'detentions_Case Threat Level',
    'detentions_Departure Country', 'detentions_Entry Status',
    'detentions_Marital', 'detentions_Stay Release Reason',
    'detentions_Detention Release Reason', 'arrests_Apprehension Method'
]

# Date columns
date_cols = [
    'detentions_Departed Date', 'detentions_Stay Book Out Date',
    'detentions_Detention Book Out Date', 'detentions_Detention Book In Date',
    'detentions_Stay Book In Date',
]

# Ensure all are datetime
for col in date_cols:
    df_cleaned[col] = pd.to_datetime(df_cleaned[col], errors='coerce')

# Numeric columns
numeric_cols = ['detentions_Birth Year']

# Fill categorical with 'Unknown'
for col in categorical_cols:
    df_cleaned[col] = df_cleaned[col].fillna('Unknown')

# Fill date columns with placeholder
for col in date_cols:
    df_cleaned[col] = df_cleaned[col].fillna(pd.Timestamp('1900-01-01'))

# Fill numeric column with median
for col in numeric_cols:
    median_val = df_cleaned[col].median()
    df_cleaned[col] = df_cleaned[col].fillna(median_val)


In [196]:
# Drop rows where 'detentions_Birth Year' or 'detentions_Case Category' are missing
df_cleaned = df_cleaned.dropna(subset=['detentions_Birth Year', 'detentions_Case Category'])

# Quick check
print("Rows remaining:", df_cleaned.shape[0])
print("Missing values in Birth Year and Case Category:")
print(df_cleaned[['detentions_Birth Year', 'detentions_Case Category']].isnull().sum())

Rows remaining: 2194210
Missing values in Birth Year and Case Category:
detentions_Birth Year       0
detentions_Case Category    0
dtype: int64


In [197]:
df_cleaned.isna().sum().sort_values(ascending=False)

detentions_Case Category               0
detentions_Detention Book Out Date     0
year                                   0
detentions_Case Status                 0
detentions_Gender                      0
detentions_Departed Date               0
detentions_Birth Year                  0
detentions_Stay Book Out Date          0
detentions_Detention Book In Date      0
detentions_Case Threat Level           0
detentions_Entry Status                0
detentions_Marital                     0
detentions_Stay Release Reason         0
detentions_Departure Country           0
detentions_Detention Release Reason    0
detentions_Detention Facility          0
arrests_Apprehension Method            0
detentions_Detention Facility Code     0
Anonymized Identifier                  0
detentions_Anonymized Identifier       0
detentions_Stay Book In Date           0
detentions_Ethnicity                   0
detentions_Final Order Yes No          0
detentions_Final_Order_numeric         0
dtype: int64

In [198]:
df_cleaned.head()

,detentions_Case Category,detentions_Detention Book Out Date,detentions_Final Order Yes No,detentions_Ethnicity,detentions_Stay Book In Date,detentions_Anonymized Identifier,Anonymized Identifier,detentions_Detention Facility Code,arrests_Apprehension Method,detentions_Detention Facility,...,detentions_Entry Status,detentions_Case Threat Level,detentions_Detention Book In Date,detentions_Stay Book Out Date,detentions_Birth Year,detentions_Departed Date,detentions_Gender,detentions_Case Status,year,detentions_Final_Order_numeric
0,[5F] Unable to Obtain Travel Document,2012-09-28,True,Not of Hispanic Origin,2012-09-28,d0451cac01101d02e3b5c236e6b7432e5eff2148,d0451cac01101d02e3b5c236e6b7432e5eff2148,SFRHOLD,CAP Local Incarceration,SFR HOLD ROOM,...,Refugee,1.0,2012-09-28,2012-09-28,1969.0,1900-01-01,Male,ACTIVE,2012,1
1,[8B] Excludable / Inadmissible - Under Adjudic...,2012-06-19,False,Hispanic Origin,2012-06-13,722038e8783f0111938ba418485b72719130bb42,722038e8783f0111938ba418485b72719130bb42,YORCOSC,CAP Local Incarceration,YORK COUNTY DETENTION CENTER,...,PWA Mexico,2.0,2012-06-13,2012-06-20,1979.0,1900-01-01,Male,ACTIVE,2012,0
2,[8C] Excludable / Inadmissible - Administrativ...,2013-02-11,True,Hispanic Origin,2012-12-07,f196886fb97872592bce7642b1c1ea79722672b5,f196886fb97872592bce7642b1c1ea79722672b5,BOONEKY,CAP Local Incarceration,BOONE COUNTY JAIL,...,PWA Mexico,2.0,2012-12-07,2013-02-11,1974.0,1900-01-01,Male,B-Relief Granted,2012,1
3,[5B] Removable - ICE Fugitive,2012-07-05,True,Unknown,2012-07-01,764bb6cc37b909a941aeebe9ae5f813bfe450c87,764bb6cc37b909a941aeebe9ae5f813bfe450c87,VTSTALB,Unknown,NORTHWEST STATE CORRECTIONAL CTR.,...,Non-Immigrant,Unknown,2012-07-01,2012-08-07,1974.0,1900-01-01,Male,ACTIVE,2012,1
4,[8E] Inadmissible - ICE Fugitive,2012-11-19,True,Hispanic Origin,2012-11-19,c269a353ec849af9c41c95409e30fe9b960889c7,c269a353ec849af9c41c95409e30fe9b960889c7,LOSCJCA,Unknown,LOS ANGELES COUNTY JAIL-TWIN TOWER,...,Unknown,Unknown,2012-11-19,2012-11-19,1973.0,1900-01-01,Male,ACTIVE,2012,1


### Feature Engineering
---
This step transforms raw columns into variables that are actually useful for a model to learn from. After combining the datasets, we created several new variables to support analysis and predictive modeling. These features include age, administration period, and time-based variables such as the number of days between arrest, detention, final order, and removal.

These engineered features help capture timelines and policy periods that may 
influence immigration case outcomes.

- administration:	Presidential administration at the time of detention
- detention_duration_days:	Days between Book In and Book Out
- stay_extended:	Flag if detention exceeded official Book Out date
- is_CAP_arrest:	Flag for Criminal Alien Program arrest
- is_287g_arrest:	Flag for 287(g) program arrest
- is_custodial_arrest:	Flag for custodial arrest
- case_threat_numeric:	Numeric encoding of case threat level
- is_hispanic:	Hispanic ethnicity flag
- is_male:	Male gender flag
- is_PWA:	Present Without Admission flag
- is_asylum_seeker:	Asylum seeker or refugee flag
- age_at_detention:	Age at detention
- is_juvenile:	Juvenile flag (under 18)

**Administrative Subsets**

To analyze differences in immigration outcomes across presidential administrations, 
we divided the dataset into subsets based on administration period. The administrations 
were defined based on fiscal years:

- Obama Administration: 2009-2012, 2013–2016 (have data starting 2012)
- Trump Administration: 2017–2020, 2025-2028
- Biden Administration: 2021–2024
- Trump Current Administration: 2025-2028

Creating these subsets allows us to compare trends, model outcomes separately, 
and analyze whether policy periods influenced final order outcomes.

In [199]:
def get_admin(year):
    if year <= 2016: 
        return 'Obama'
    elif 2017 <= year <= 2020: 
        return 'Trump'
    elif 2021 <= year <= 2024: 
        return 'Biden'
    elif year >= 2025: 
        return 'Trump' # Corrected for current administration
    else: 
        return 'Unknown'

df_cleaned['administration'] = df_cleaned['year'].apply(get_admin)

In [200]:
df_cleaned['administration'].value_counts().reindex(['Obama', 'Trump', 'Biden', 'Unknown'])

administration
Obama      933387.0
Trump      857732.0
Biden      403091.0
Unknown         NaN
Name: count, dtype: float64

In [201]:
print(df_cleaned.columns.tolist())

['detentions_Case Category', 'detentions_Detention Book Out Date', 'detentions_Final Order Yes No', 'detentions_Ethnicity', 'detentions_Stay Book In Date', 'detentions_Anonymized Identifier', 'Anonymized Identifier', 'detentions_Detention Facility Code', 'arrests_Apprehension Method', 'detentions_Detention Facility', 'detentions_Detention Release Reason', 'detentions_Departure Country', 'detentions_Stay Release Reason', 'detentions_Marital', 'detentions_Entry Status', 'detentions_Case Threat Level', 'detentions_Detention Book In Date', 'detentions_Stay Book Out Date', 'detentions_Birth Year', 'detentions_Departed Date', 'detentions_Gender', 'detentions_Case Status', 'year', 'detentions_Final_Order_numeric', 'administration']


**Dates/Processing Speed Features**
We calculate how long someone was held and whether they stayed beyond the official Book Out date:

In [202]:
# Detention duration and stay flags
# Define placeholder date used for missing
placeholder_date = pd.Timestamp('1900-01-01')
# Duration between Book In and Book Out, ignoring placeholder dates
df_cleaned['detention_duration_days'] = np.where(
    (df_cleaned['detentions_Detention Book In Date'] == placeholder_date) |
    (df_cleaned['detentions_Detention Book Out Date'] == placeholder_date),
    np.nan,  # set duration to NaN if either date is missing
    (df_cleaned['detentions_Detention Book Out Date'] - df_cleaned['detentions_Detention Book In Date']).dt.days
)

# Stay-related flags: were they held past a certain point
df_cleaned['stay_extended'] = np.where(df_cleaned['detentions_Stay Book Out Date'] > df_cleaned['detentions_Detention Book Out Date'], 1, 0)

# Quick check
print(df_cleaned[['detention_duration_days','stay_extended']].head())

   detention_duration_days  stay_extended
0                      0.0              0
1                      6.0              1
2                     66.0              0
3                      4.0              1
4                      0.0              0


**Criminality Features**

We create flags for different types of arrests and criminal justice entanglements:

In [203]:
df_cleaned['arrests_Apprehension Method'].unique()

array(['CAP Local Incarceration', 'Unknown', 'CAP State Incarceration',
       'Law Enforcement Agency Response Unit', '287(g) Program',
       'Located', 'ERO Reprocessed Arrest', 'Non-Custodial Arrest',
       'CAP Federal Incarceration', 'Other Agency (turned over to INS)',
       'Other efforts', 'Inspections', 'Other Task Force',
       'Patrol Border', 'Organized Crime Drug Enforcement Task Force',
       'Anti-Smuggling', 'Patrol Interior', 'Worksite Enforcement',
       'Boat Patrol', 'Transportation Check Aircraft',
       'Probation and Parole', 'Crewman/Stowaway',
       'Transportation Check Freight Train',
       'Transportation Check Passenger Train', 'Traffic Check',
       'Foreign Fugitive', 'Criminal Alien Program', 'Custodial Arrest'],
      dtype=object)

In [204]:
df_cleaned['detentions_Case Threat Level'].unique()

array([1.0, 2.0, 'Unknown', 3.0], dtype=object)

In [205]:
# Criminal justice / detention flags
df_cleaned['is_CAP_arrest'] = df_cleaned['arrests_Apprehension Method'].str.contains('CAP', na=False).astype(int)
df_cleaned['is_287g_arrest'] = df_cleaned['arrests_Apprehension Method'].str.contains('287', na=False).astype(int)
df_cleaned['is_custodial_arrest'] = (df_cleaned['arrests_Apprehension Method'] == 'Custodial Arrest').astype(int)

# Case complexity / threat
df_cleaned['case_threat_numeric'] = df_cleaned['detentions_Case Threat Level'].replace('Unknown', 0).astype(float)

# Quick check
print(df_cleaned[['is_CAP_arrest','is_287g_arrest','is_custodial_arrest', 'case_threat_numeric']].head())

   is_CAP_arrest  is_287g_arrest  is_custodial_arrest  case_threat_numeric
0              1               0                    0                  1.0
1              1               0                    0                  2.0
2              1               0                    0                  2.0
3              0               0                    0                  0.0
4              0               0                    0                  0.0


/var/folders/_s/x14mjmjn7qj5qh8bhxln436r0000gn/T/ipykernel_34384/2279668919.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cleaned['case_threat_numeric'] = df_cleaned['detentions_Case Threat Level'].replace('Unknown', 0).astype(float)


**Case/Detention Complexity Features**

We simplify sparse categorical variables into binary flags for modeling and calculate age at detention and flag juveniles:

In [206]:
df_cleaned['detentions_Ethnicity'].unique()

array(['Not of Hispanic Origin', 'Hispanic Origin', 'Unknown'],
      dtype=object)

In [207]:
df_cleaned['detentions_Gender'].unique()

array(['Male', 'Female', 'Unknown'], dtype=object)

In [208]:
df_cleaned['detentions_Entry Status'].unique()

array(['Refugee', 'PWA Mexico', 'Non-Immigrant', 'Unknown',
       'Legal Permanent Resident', 'PWA Other', 'Border Crossing Card',
       'Immigrant', 'Parolee', 'Asylum', 'Student', 'Visa Waiver Program',
       'Visitor', 'Other', 'False Claim with Counterfeit Document',
       'PWA Canada', 'Not  Applicable', 'Imposter', 'Absconder',
       'Conditional Resident', 'Crew',
       'False Claim with Altered Document', 'Temporary Worker Other',
       'False Claim with Valid Document', 'Stowaway',
       'Oral False Claim to U.S. Citizenship',
       'Temporary Work Agriculture', 'Temporary Resident',
       'ORAL FALSE CLAIMS TO OTHER THAN U.S. CITZ-', 'Not in Custody',
       'Smuggler', 'TWOV', 'Guam Visa Waiver',
       'Lawful Permanent Resident - Seeking admission (OALICE)',
       'US Citizen', 'Other Non-Immigrant Classification', 'No Documents',
       'Present Without Admission',
       'Lawful Permanent Resident - Not seeking admission',
       'Crew - Deserter', 'Initial En

In [209]:
# Ethnicity -> is_hispanic
df_cleaned['is_hispanic'] = df_cleaned['detentions_Ethnicity'].apply(
    lambda x: 1 if x == 'Hispanic Origin' else 0
)

# Gender -> is_male
df_cleaned['is_male'] = df_cleaned['detentions_Gender'].apply(
    lambda x: 1 if x == 'Male' else 0
)

# Entry status -> is_PWA (Present Without Authorization)
df_cleaned['is_PWA'] = df_cleaned['detentions_Entry Status'].apply(
    lambda x: 1 if isinstance(x, str) and 'PWA' in x else 0
)

# Flag if the person is an asylum seeker or refugee
df_cleaned['is_asylum_seeker'] = df_cleaned['detentions_Entry Status'].str.contains(
    'Asylum|Refugee', case=False, na=False
).astype(int)

# Compute age at detention
df_cleaned['age_at_detention'] = df_cleaned['year'] - df_cleaned['detentions_Birth Year']

# Flag if juvenile (under 18)
df_cleaned['is_juvenile'] = (df_cleaned['age_at_detention'] < 18).astype(int)

# Quick check
print(df_cleaned[['is_hispanic','is_male','is_PWA', 'is_asylum_seeker', 'detentions_Birth Year', 'year', 'age_at_detention', 'is_juvenile']].head())

   is_hispanic  is_male  is_PWA  is_asylum_seeker  detentions_Birth Year  \
0            0        1       0                 1                 1969.0   
1            1        1       1                 0                 1979.0   
2            1        1       1                 0                 1974.0   
3            0        1       0                 0                 1974.0   
4            1        1       0                 0                 1973.0   

   year  age_at_detention  is_juvenile  
0  2012              43.0            0  
1  2012              33.0            0  
2  2012              38.0            0  
3  2012              38.0            0  
4  2012              39.0            0  


### Final Modeling Datasets
---

#### Master Dataset

In [210]:
# master dataset ready for modeling
df_cleaned.head()

,detentions_Case Category,detentions_Detention Book Out Date,detentions_Final Order Yes No,detentions_Ethnicity,detentions_Stay Book In Date,detentions_Anonymized Identifier,Anonymized Identifier,detentions_Detention Facility Code,arrests_Apprehension Method,detentions_Detention Facility,...,is_CAP_arrest,is_287g_arrest,is_custodial_arrest,case_threat_numeric,is_hispanic,is_male,is_PWA,is_asylum_seeker,age_at_detention,is_juvenile
0,[5F] Unable to Obtain Travel Document,2012-09-28,True,Not of Hispanic Origin,2012-09-28,d0451cac01101d02e3b5c236e6b7432e5eff2148,d0451cac01101d02e3b5c236e6b7432e5eff2148,SFRHOLD,CAP Local Incarceration,SFR HOLD ROOM,...,1,0,0,1.0,0,1,0,1,43.0,0
1,[8B] Excludable / Inadmissible - Under Adjudic...,2012-06-19,False,Hispanic Origin,2012-06-13,722038e8783f0111938ba418485b72719130bb42,722038e8783f0111938ba418485b72719130bb42,YORCOSC,CAP Local Incarceration,YORK COUNTY DETENTION CENTER,...,1,0,0,2.0,1,1,1,0,33.0,0
2,[8C] Excludable / Inadmissible - Administrativ...,2013-02-11,True,Hispanic Origin,2012-12-07,f196886fb97872592bce7642b1c1ea79722672b5,f196886fb97872592bce7642b1c1ea79722672b5,BOONEKY,CAP Local Incarceration,BOONE COUNTY JAIL,...,1,0,0,2.0,1,1,1,0,38.0,0
3,[5B] Removable - ICE Fugitive,2012-07-05,True,Unknown,2012-07-01,764bb6cc37b909a941aeebe9ae5f813bfe450c87,764bb6cc37b909a941aeebe9ae5f813bfe450c87,VTSTALB,Unknown,NORTHWEST STATE CORRECTIONAL CTR.,...,0,0,0,0.0,0,1,0,0,38.0,0
4,[8E] Inadmissible - ICE Fugitive,2012-11-19,True,Hispanic Origin,2012-11-19,c269a353ec849af9c41c95409e30fe9b960889c7,c269a353ec849af9c41c95409e30fe9b960889c7,LOSCJCA,Unknown,LOS ANGELES COUNTY JAIL-TWIN TOWER,...,0,0,0,0.0,1,1,0,0,39.0,0


In [211]:

print("Columns in the master dataset:")
print(df_cleaned.columns.tolist())

Columns in the master dataset:
['detentions_Case Category', 'detentions_Detention Book Out Date', 'detentions_Final Order Yes No', 'detentions_Ethnicity', 'detentions_Stay Book In Date', 'detentions_Anonymized Identifier', 'Anonymized Identifier', 'detentions_Detention Facility Code', 'arrests_Apprehension Method', 'detentions_Detention Facility', 'detentions_Detention Release Reason', 'detentions_Departure Country', 'detentions_Stay Release Reason', 'detentions_Marital', 'detentions_Entry Status', 'detentions_Case Threat Level', 'detentions_Detention Book In Date', 'detentions_Stay Book Out Date', 'detentions_Birth Year', 'detentions_Departed Date', 'detentions_Gender', 'detentions_Case Status', 'year', 'detentions_Final_Order_numeric', 'administration', 'detention_duration_days', 'stay_extended', 'is_CAP_arrest', 'is_287g_arrest', 'is_custodial_arrest', 'case_threat_numeric', 'is_hispanic', 'is_male', 'is_PWA', 'is_asylum_seeker', 'age_at_detention', 'is_juvenile']


In [ ]:
# Save the cleaned master DataFrame to a CSV
# df_cleaned.to_csv("ICE_Master_Cleaned_withFeatures.csv", index=False)
# print(f"Cleaned master saved: ICE_Master_Cleaned_withFeatures.csv ({len(df_cleaned)} rows, {len(df_cleaned.columns)} columns)")

Cleaned master saved: ICE_Master_Cleaned.csv (2194210 rows, 37 columns)


#### Administration Subsets

In [ ]:
cols_to_export = df_cleaned.columns.tolist()  # all columns

for admin in ['Obama', 'Trump', 'Biden']:
    subset = df_cleaned[df_cleaned['administration'] == admin][cols_to_export]
    print(f"--- Head of {admin} subset ({len(subset)} rows) ---")
    display(subset.head())  # show first 5 rows in Jupyter

    # code to download subsets as CSV files
    # filename = f"ICE_{admin}_subset.csv"
    # subset.to_csv(filename, index=False)
    # print(f"Saved {filename}\n")

--- Head of Obama subset (933387 rows) ---


,detentions_Case Category,detentions_Detention Book Out Date,detentions_Final Order Yes No,detentions_Ethnicity,detentions_Stay Book In Date,detentions_Anonymized Identifier,Anonymized Identifier,detentions_Detention Facility Code,arrests_Apprehension Method,detentions_Detention Facility,...,is_CAP_arrest,is_287g_arrest,is_custodial_arrest,case_threat_numeric,is_hispanic,is_male,is_PWA,is_asylum_seeker,age_at_detention,is_juvenile
0,[5F] Unable to Obtain Travel Document,2012-09-28,True,Not of Hispanic Origin,2012-09-28,d0451cac01101d02e3b5c236e6b7432e5eff2148,d0451cac01101d02e3b5c236e6b7432e5eff2148,SFRHOLD,CAP Local Incarceration,SFR HOLD ROOM,...,1,0,0,1.0,0,1,0,1,43.0,0
1,[8B] Excludable / Inadmissible - Under Adjudic...,2012-06-19,False,Hispanic Origin,2012-06-13,722038e8783f0111938ba418485b72719130bb42,722038e8783f0111938ba418485b72719130bb42,YORCOSC,CAP Local Incarceration,YORK COUNTY DETENTION CENTER,...,1,0,0,2.0,1,1,1,0,33.0,0
2,[8C] Excludable / Inadmissible - Administrativ...,2013-02-11,True,Hispanic Origin,2012-12-07,f196886fb97872592bce7642b1c1ea79722672b5,f196886fb97872592bce7642b1c1ea79722672b5,BOONEKY,CAP Local Incarceration,BOONE COUNTY JAIL,...,1,0,0,2.0,1,1,1,0,38.0,0
3,[5B] Removable - ICE Fugitive,2012-07-05,True,Unknown,2012-07-01,764bb6cc37b909a941aeebe9ae5f813bfe450c87,764bb6cc37b909a941aeebe9ae5f813bfe450c87,VTSTALB,Unknown,NORTHWEST STATE CORRECTIONAL CTR.,...,0,0,0,0.0,0,1,0,0,38.0,0
4,[8E] Inadmissible - ICE Fugitive,2012-11-19,True,Hispanic Origin,2012-11-19,c269a353ec849af9c41c95409e30fe9b960889c7,c269a353ec849af9c41c95409e30fe9b960889c7,LOSCJCA,Unknown,LOS ANGELES COUNTY JAIL-TWIN TOWER,...,0,0,0,0.0,1,1,0,0,39.0,0


Saved ICE_Obama_subset.csv

--- Head of Trump subset (857732 rows) ---


,detentions_Case Category,detentions_Detention Book Out Date,detentions_Final Order Yes No,detentions_Ethnicity,detentions_Stay Book In Date,detentions_Anonymized Identifier,Anonymized Identifier,detentions_Detention Facility Code,arrests_Apprehension Method,detentions_Detention Facility,...,is_CAP_arrest,is_287g_arrest,is_custodial_arrest,case_threat_numeric,is_hispanic,is_male,is_PWA,is_asylum_seeker,age_at_detention,is_juvenile
935939,[8B] Excludable / Inadmissible - Under Adjudic...,2017-11-26,False,Unknown,2017-11-12,48642ead9e3364cdce3f29a23a4f9399927e4fed,48642ead9e3364cdce3f29a23a4f9399927e4fed,SLRDCAZ,Unknown,SAN LUIS REGIONAL DET CENTER,...,0,0,0,0.0,0,1,1,0,21.0,0
935940,[3] Deportable - Administratively Final Order,2018-01-11,True,Hispanic Origin,2017-12-14,fc386240402229034c0fcb11c2ce89cfb82269dc,fc386240402229034c0fcb11c2ce89cfb82269dc,EPC,Unknown,EL PASO SPC,...,0,0,0,0.0,1,0,0,0,44.0,0
935941,[2A] Deportable - Under Adjudication by IJ,2018-02-23,False,Hispanic Origin,2017-12-04,fa2ce674e31fb126d2f65e336cdd1f4956e45c81,fa2ce674e31fb126d2f65e336cdd1f4956e45c81,GLADEFL,Unknown,GLADES COUNTY DETENTION CENTER,...,0,0,0,2.0,1,1,0,0,36.0,0
935942,[8F] Expedited Removal,2017-10-19,True,Unknown,2017-10-19,434f3154aa599c833c1a3ac140f685326a671256,434f3154aa599c833c1a3ac140f685326a671256,PCSHOLD,Unknown,PECOS HOLD ROOM,...,0,0,0,3.0,0,1,1,0,48.0,0
935943,[8C] Excludable / Inadmissible - Administrativ...,2018-02-14,True,Hispanic Origin,2017-10-27,91a7fb2a3532a68463359f855acd24bd694dd14b,91a7fb2a3532a68463359f855acd24bd694dd14b,GLADEFL,Unknown,GLADES COUNTY DETENTION CENTER,...,0,0,0,2.0,1,1,0,0,35.0,0


Saved ICE_Trump_subset.csv

--- Head of Biden subset (403091 rows) ---


,detentions_Case Category,detentions_Detention Book Out Date,detentions_Final Order Yes No,detentions_Ethnicity,detentions_Stay Book In Date,detentions_Anonymized Identifier,Anonymized Identifier,detentions_Detention Facility Code,arrests_Apprehension Method,detentions_Detention Facility,...,is_CAP_arrest,is_287g_arrest,is_custodial_arrest,case_threat_numeric,is_hispanic,is_male,is_PWA,is_asylum_seeker,age_at_detention,is_juvenile
1756894,[8G] Expedited Removal - Credible Fear Referral,2021-08-13,False,Unknown,2021-08-06,d9a67a82d6f078a5591424caf84b7d67d21ef388,d9a67a82d6f078a5591424caf84b7d67d21ef388,CCAHUTX,Unknown,T DON HUTTO DETENTION CENTER,...,0,0,0,0.0,0,0,1,0,25.0,0
1756895,[8B] Excludable / Inadmissible - Under Adjudic...,2021-06-03,False,Unknown,2021-06-01,5e6ad00ac1ca41ba7a9ca72955383db7d79e4e60,5e6ad00ac1ca41ba7a9ca72955383db7d79e4e60,ALESSAZ,Unknown,STES ON SCOTTSDALE CASA DE ALEGRIA,...,0,0,0,0.0,0,1,1,0,33.0,0
1756896,[8A] Excludable / Inadmissible - Hearing Not C...,2021-12-08,False,Unknown,2021-12-01,1d95d2f193ef2338ebc168a78144e8fff0a555ae,1d95d2f193ef2338ebc168a78144e8fff0a555ae,PRLDCTX,Unknown,PRAIRIELAND DETENTION CENTER,...,0,0,0,0.0,0,1,1,0,35.0,0
1756897,[8A] Excludable / Inadmissible - Hearing Not C...,2021-08-11,False,Unknown,2021-07-25,2c89d9fe27bfad63e49262549321f7dc8bc78fb4,2c89d9fe27bfad63e49262549321f7dc8bc78fb4,BLBNATX,Unknown,BLUEBONNET DET FCLTY,...,0,0,0,0.0,0,1,1,0,26.0,0
1756898,[8A] Excludable / Inadmissible - Hearing Not C...,2021-09-05,False,Unknown,2021-09-05,2238e4080589810793b4ce5240bc36530c27691f,2238e4080589810793b4ce5240bc36530c27691f,EPCPCTX,Non-Custodial Arrest,EGP CPC HOLDING,...,0,0,0,0.0,0,1,1,0,11.0,1


Saved ICE_Biden_subset.csv



## Exploratory Data Analysis

In [214]:
# EDA library import
import matplotlib.pyplot as plt
import seaborn as sns

### EDA of Deportation Data from 2012 - 2025
--- 

In [217]:
master_df = pd.read_csv("../data/ICE_Master_Cleaned_withFeatures.csv")

# Basic info
print("Dataset Shape:", master_df.shape)
print("\nSample Data:\n", master_df.head())

Dataset Shape: (2194210, 37)

Sample Data:
                             detentions_Case Category  \
0              [5F] Unable to Obtain Travel Document   
1  [8B] Excludable / Inadmissible - Under Adjudic...   
2  [8C] Excludable / Inadmissible - Administrativ...   
3                      [5B] Removable - ICE Fugitive   
4                   [8E] Inadmissible - ICE Fugitive   

  detentions_Detention Book Out Date  detentions_Final Order Yes No  \
0                         2012-09-28                           True   
1                         2012-06-19                          False   
2                         2013-02-11                           True   
3                         2012-07-05                           True   
4                         2012-11-19                           True   

     detentions_Ethnicity detentions_Stay Book In Date  \
0  Not of Hispanic Origin                   2012-09-28   
1         Hispanic Origin                   2012-06-13   
2         Hispanic

### EDA of Deportation Data from Obama's Administration
---

In [218]:
obama_df = pd.read_csv("../data/ICE_Obama_subset.csv")

obama_df.head()

,detentions_Case Category,detentions_Detention Book Out Date,detentions_Final Order Yes No,detentions_Ethnicity,detentions_Stay Book In Date,detentions_Anonymized Identifier,Anonymized Identifier,detentions_Detention Facility Code,arrests_Apprehension Method,detentions_Detention Facility,...,is_CAP_arrest,is_287g_arrest,is_custodial_arrest,case_threat_numeric,is_hispanic,is_male,is_PWA,is_asylum_seeker,age_at_detention,is_juvenile
0,[5F] Unable to Obtain Travel Document,2012-09-28,True,Not of Hispanic Origin,2012-09-28,d0451cac01101d02e3b5c236e6b7432e5eff2148,d0451cac01101d02e3b5c236e6b7432e5eff2148,SFRHOLD,CAP Local Incarceration,SFR HOLD ROOM,...,1,0,0,1.0,0,1,0,1,43.0,0
1,[8B] Excludable / Inadmissible - Under Adjudic...,2012-06-19,False,Hispanic Origin,2012-06-13,722038e8783f0111938ba418485b72719130bb42,722038e8783f0111938ba418485b72719130bb42,YORCOSC,CAP Local Incarceration,YORK COUNTY DETENTION CENTER,...,1,0,0,2.0,1,1,1,0,33.0,0
2,[8C] Excludable / Inadmissible - Administrativ...,2013-02-11,True,Hispanic Origin,2012-12-07,f196886fb97872592bce7642b1c1ea79722672b5,f196886fb97872592bce7642b1c1ea79722672b5,BOONEKY,CAP Local Incarceration,BOONE COUNTY JAIL,...,1,0,0,2.0,1,1,1,0,38.0,0
3,[5B] Removable - ICE Fugitive,2012-07-05,True,Unknown,2012-07-01,764bb6cc37b909a941aeebe9ae5f813bfe450c87,764bb6cc37b909a941aeebe9ae5f813bfe450c87,VTSTALB,Unknown,NORTHWEST STATE CORRECTIONAL CTR.,...,0,0,0,0.0,0,1,0,0,38.0,0
4,[8E] Inadmissible - ICE Fugitive,2012-11-19,True,Hispanic Origin,2012-11-19,c269a353ec849af9c41c95409e30fe9b960889c7,c269a353ec849af9c41c95409e30fe9b960889c7,LOSCJCA,Unknown,LOS ANGELES COUNTY JAIL-TWIN TOWER,...,0,0,0,0.0,1,1,0,0,39.0,0


### EDA of Deportation Data from Trumps's Administration
---

In [219]:
trump_df = pd.read_csv("../data/ICE_Trump_subset.csv")

trump_df.head()

,detentions_Case Category,detentions_Detention Book Out Date,detentions_Final Order Yes No,detentions_Ethnicity,detentions_Stay Book In Date,detentions_Anonymized Identifier,Anonymized Identifier,detentions_Detention Facility Code,arrests_Apprehension Method,detentions_Detention Facility,...,is_CAP_arrest,is_287g_arrest,is_custodial_arrest,case_threat_numeric,is_hispanic,is_male,is_PWA,is_asylum_seeker,age_at_detention,is_juvenile
0,[8B] Excludable / Inadmissible - Under Adjudic...,2017-11-26,False,Unknown,2017-11-12,48642ead9e3364cdce3f29a23a4f9399927e4fed,48642ead9e3364cdce3f29a23a4f9399927e4fed,SLRDCAZ,Unknown,SAN LUIS REGIONAL DET CENTER,...,0,0,0,0.0,0,1,1,0,21.0,0
1,[3] Deportable - Administratively Final Order,2018-01-11,True,Hispanic Origin,2017-12-14,fc386240402229034c0fcb11c2ce89cfb82269dc,fc386240402229034c0fcb11c2ce89cfb82269dc,EPC,Unknown,EL PASO SPC,...,0,0,0,0.0,1,0,0,0,44.0,0
2,[2A] Deportable - Under Adjudication by IJ,2018-02-23,False,Hispanic Origin,2017-12-04,fa2ce674e31fb126d2f65e336cdd1f4956e45c81,fa2ce674e31fb126d2f65e336cdd1f4956e45c81,GLADEFL,Unknown,GLADES COUNTY DETENTION CENTER,...,0,0,0,2.0,1,1,0,0,36.0,0
3,[8F] Expedited Removal,2017-10-19,True,Unknown,2017-10-19,434f3154aa599c833c1a3ac140f685326a671256,434f3154aa599c833c1a3ac140f685326a671256,PCSHOLD,Unknown,PECOS HOLD ROOM,...,0,0,0,3.0,0,1,1,0,48.0,0
4,[8C] Excludable / Inadmissible - Administrativ...,2018-02-14,True,Hispanic Origin,2017-10-27,91a7fb2a3532a68463359f855acd24bd694dd14b,91a7fb2a3532a68463359f855acd24bd694dd14b,GLADEFL,Unknown,GLADES COUNTY DETENTION CENTER,...,0,0,0,2.0,1,1,0,0,35.0,0
